# 08b — Skalierungs-Faithfulness (Phase 3b)

Ground-truth-verankerte Faithfulness-Metriken nach **Ichmoukhamedov et al. (2024)**
(RA/SA/VA, Gl. 1) auf **n≈200 Instanzen × N Generationen** — das objektivste
Faithfulness-Maß (kein LLM-Judge-Bias). Treuer Port von `08_Evaluation_Ichmoukhamedov`
(n=20, unangetastet) über `utils.faithfulness`; gen-aware, Batch-Extraktion (−50 %).

**Vorsicht — Extraktionsvalidität (Abschnitt 3):** Der Extraktor ist selbst
fehleranfällig (NB 09: 19/30 Extraktor-Artefakte, Rang-0-Genauigkeit 55 %). RA/SA/VA
messen **Präzision, nicht Recall** (NB 08 §4.1). Daher werden Top-K-Recall,
Out-of-Top-K-Rate, Parse-Ausfallrate und Rang-0-Trefferquote des Extraktors
**mitberichtet** und ein stratifiziertes Hand-Check-Subsample für die manuelle
Validierung (NB-09-Kodierung auf n≈200 erweitern) exportiert.

**Voraussetzungen:** NB 04/05/06 mit `SCALE_RUN=True` durchgelaufen
(`results/pipeline0X/scale/`); `ANTHROPIC_API_KEY` gesetzt.

In [ ]:
from __future__ import annotations

import sys, json
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from IPython.display import display

from utils import scale_instance_ids, N_GENERATIONS_SCALE, RESULTS_DIR
from utils.eval import load_scale_records, PIPELINE_LABELS
from utils.faithfulness import (
    EXTRACTION_SYSTEM, TOP_K_DEFAULT, build_extraction_prompt, parse_extraction,
    extraction_base_cid, build_faithfulness_df, extraction_validity_summary,
)
from utils.batch import message_request, run_batch
from utils.llm import build_text_params, DEFAULT_MODEL
from utils.stats import bootstrap_ci, wilcoxon_pairwise

# ── Konfiguration ────────────────────────────────────────────────
LOSS_KEY        = 'poisson_log'
PIPELINES       = ['04', '05', '06']            # wie NB 08 (Template 00 ist deterministisch)
XAI_MODELS      = ['xgb', 'ebm']
PIP_ORDER       = [PIPELINE_LABELS[p] for p in PIPELINES]
PIPELINE_COLORS = {'JSON→Text': '#4c72b0', 'Vision': '#dd8452', 'Tool-Use': '#55a868'}
METRICS         = ['RA', 'SA', 'VA']

EXTRACT_MODEL      = DEFAULT_MODEL   # Extraction LLM — identisch zu NB 08 (Sonnet)
EXTRACT_MAX_TOKENS = 700             # wie NB 08
EXTRACT_PREFIX     = 'ext'           # custom_id-Prefix der Extraktions-Requests
TOP_K              = TOP_K_DEFAULT   # Paper: top-4

INSTANCE_IDS_SCALE = scale_instance_ids()
N_GEN              = N_GENERATIONS_SCALE

# Faithfulness-Artefakte des Skalierungslaufs — getrennt von der n=20-Validität (NB 08).
SCALE_EVAL_DIR = RESULTS_DIR / 'scale_eval'
FAITH_DIR      = SCALE_EVAL_DIR / 'faithfulness'
FAITH_DIR.mkdir(parents=True, exist_ok=True)

print(f'Instanzen:    {len(INSTANCE_IDS_SCALE)} (seeded, stratifiziert)')
print(f'Pipelines:    {PIP_ORDER}')
print(f'Generationen: 04/05/06 → {N_GEN}')
print(f'Extraktor:    {EXTRACT_MODEL} (Batch, −50 %)  Top-K={TOP_K}')
print(f'Ausgabe:      {FAITH_DIR}')

## 1 · Generierungs-Artefakte laden (gen-aware)

In [ ]:
df_scale = load_scale_records(
    PIPELINES, XAI_MODELS, INSTANCE_IDS_SCALE, N_GEN,
    results_dir=RESULTS_DIR, loss_key=LOSS_KEY, require_complete=False,
)
print(f'{len(df_scale)} Narrative geladen.')
display(df_scale.groupby(['pipeline_label', 'xai_model']).size().unstack())

## 2 · Extraktion (Batch, −50 %)

Das Extraction LLM (Sonnet, identisch zu NB 08) extrahiert pro Narrativ je Feature
Rang/Vorzeichen/Wert/Annahme. Einzelschritt-Call → voll batchbar. Ergebnisse werden
gecacht (`extractions_scale.json`); `state_path` erlaubt Poll-Resume nach Abbruch.

In [ ]:
ext_cache_path = FAITH_DIR / 'extractions_scale.json'

def _extraction_requests(df):
    """Baut (message_request, custom_id) je Narrativ — gen-aware."""
    requests, cids = [], []
    for _, row in df.iterrows():
        cid = extraction_base_cid(EXTRACT_PREFIX, row['pipeline'], row['xai_model'],
                                  int(row['instance_id']), int(row['generation']))
        prompt = build_extraction_prompt(row['explanation'], row['xai_model'])
        params = build_text_params(prompt, system=EXTRACTION_SYSTEM, model=EXTRACT_MODEL,
                                   max_tokens=EXTRACT_MAX_TOKENS, cache_system=True)
        requests.append(message_request(cid, params))
        cids.append(cid)
    return requests, cids

extraction_by_cid = json.loads(ext_cache_path.read_text()) if ext_cache_path.exists() else {}
requests, cids = _extraction_requests(df_scale)

if all(c in extraction_by_cid for c in cids):
    print(f'Alle {len(cids)} Extraktionen aus Cache: {ext_cache_path.name}')
else:
    print(f'Reiche {len(requests)} Extraktionen als Batch ein ({EXTRACT_MODEL}, −50 %) …')
    out = run_batch(requests, parse=parse_extraction,
                    state_path=FAITH_DIR / '_extraction_scale_state.json')
    extraction_by_cid = out['succeeded']
    if out['failed']:
        print(f"⚠️  {len(out['failed'])} Extraktionen endgültig fehlgeschlagen "
              f'(werden als leer = parse_empty gewertet).')
    ext_cache_path.write_text(json.dumps(extraction_by_cid, indent=2, ensure_ascii=False))
    print(f'Gespeichert: {ext_cache_path}')

n_empty = sum(1 for c in cids if not extraction_by_cid.get(c))
print(f'Extraktionen vorhanden: {len(cids) - n_empty}/{len(cids)} (leer/fehlend: {n_empty})')

## 3 · Faithfulness RA/SA/VA (Gl. 1) + Bootstrap-CI

$$X_A = \frac{\sum_{x_j \neq \phi} \delta_{x_j, x_j^*}}{n - \sum 1[x_j = \phi]}$$

Jede Narrative-Generation ist eine Beobachtung. `None` (kein extrahiertes Feature
unter Top-K) wird aus dem CI ausgeschlossen.

In [ ]:
faith_df = build_faithfulness_df(
    df_scale, extraction_by_cid, prefix=EXTRACT_PREFIX, top_k=TOP_K, loss_key=LOSS_KEY,
)
faith_df.to_csv(FAITH_DIR / 'faithfulness_scale_metrics.csv', index=False)

summary = faith_df.groupby('pipeline_label')[METRICS].mean().round(3).reindex(PIP_ORDER)
print(f'Ø RA/SA/VA je Pipeline (n≈{len(INSTANCE_IDS_SCALE)} × {N_GEN} Gen.):')
display(summary)
summary.to_csv(FAITH_DIR / 'faithfulness_scale_summary.csv')

ci_rows = []
for pipeline in PIP_ORDER:
    sub = faith_df[faith_df.pipeline_label == pipeline]
    entry = {'pipeline': pipeline}
    for m in METRICS:
        vals = pd.to_numeric(sub[m], errors='coerce').dropna().values
        lo, hi, obs = bootstrap_ci(vals)
        entry[m] = f'{obs:.2f} [{lo:.2f}, {hi:.2f}]' if not np.isnan(obs) else 'n/a'
    ci_rows.append(entry)
ci_df = pd.DataFrame(ci_rows).set_index('pipeline')
print('\n95 %-Bootstrap-CI je Metrik:')
display(ci_df)
ci_df.to_csv(FAITH_DIR / 'faithfulness_scale_bootstrap_ci.csv')

## 4 · Extraktionsvalidität (Verlässlichkeit des Messinstruments)

RA/SA/VA werten nur die *erwähnten* Features (Präzision). Diese Kennzahlen machen
sichtbar, wie verlässlich der **Extraktor** ist — zwingend, weil er fehleranfällig ist:

- **parse_empty_rate** — Anteil Narrative ohne gültige JSON-Extraktion (Messausfall).
- **mean_topk_recall** — Anteil der Top-K-Ground-Truth-Features, die erfasst wurden
  (Gegenstück zur Präzision von RA/SA/VA — NB 08 §4.1).
- **out_of_topk_rate** — Anteil extrahierter Features außerhalb Top-K (Rausch-Proxy).
- **r0_match_rate** — Rang-0-Trefferquote Extraktor ↔ SHAP (Bezug NB 09: 55 %).

Zusätzlich wird ein stratifiziertes **Hand-Check-Subsample** exportiert, um die
NB-09-Extraktor-Kodierung (E1/E2/E3 = Mess-Artefakt; B1/B2/C = Erklärungsfehler) auf
den größeren Lauf auszuweiten.

In [ ]:
valid_summary = extraction_validity_summary(faith_df).reindex(PIP_ORDER)
print('Extraktionsvalidität je Pipeline — RA/SA/VA nur im Licht dieser Coverage lesen:')
display(valid_summary)
valid_summary.to_csv(FAITH_DIR / 'extraction_validity_summary.csv')

# Stratifiziertes Hand-Check-Subsample (je Pipeline) für die manuelle Kodierung.
HANDCHECK_PER_PIPELINE = 10
hc = (faith_df.groupby('pipeline_label', group_keys=False)
              .apply(lambda g: g.sample(min(HANDCHECK_PER_PIPELINE, len(g)),
                                        random_state=42)))
hc_cols = ['pipeline', 'pipeline_label', 'xai_model', 'instance_id', 'generation',
           'RA', 'SA', 'VA', 'n_extracted', 'topk_recall', 'n_out_of_topk',
           'gt_r0_feature', 'ext_r0_feature', 'r0_match']
hc = hc[hc_cols].reset_index(drop=True)

# Rohextraktion + Narrativ für die Kodierung beilegen.
expl_lookup = {extraction_base_cid(EXTRACT_PREFIX, r['pipeline'], r['xai_model'],
                                   int(r['instance_id']), int(r['generation'])): r['explanation']
               for _, r in df_scale.iterrows()}
hc['raw_extraction'] = [
    json.dumps(extraction_by_cid.get(
        extraction_base_cid(EXTRACT_PREFIX, r['pipeline'], r['xai_model'],
                            int(r['instance_id']), int(r['generation'])), {}),
        ensure_ascii=False)
    for _, r in hc.iterrows()
]
hc['explanation'] = [
    expl_lookup.get(extraction_base_cid(EXTRACT_PREFIX, r['pipeline'], r['xai_model'],
                                        int(r['instance_id']), int(r['generation'])), '')
    for _, r in hc.iterrows()
]
hc['manual_code'] = ''   # leer → manuell kodieren (E1/E2/E3/B1/B2/C wie NB 09)
hc.to_csv(FAITH_DIR / 'extraction_handcheck_sample.csv', index=False)
print(f'\nHand-Check-Subsample: {len(hc)} Fälle → {FAITH_DIR / "extraction_handcheck_sample.csv"}')
print('Spalte manual_code leer — manuell kodieren, dann Mess- vs. Erklärungsanteil berichten.')

## 5 · Paarweise Wilcoxon + Cliff's δ

Die N Generationen je (Instanz × Modell) werden gemittelt → ein gepaarter Wert je
Einheit über alle Pipelines. RA/SA/VA-`None` fließt als NaN ein und wird gepaart entfernt.

In [ ]:
df_agg = faith_df.copy()
for m in METRICS:
    df_agg[m] = pd.to_numeric(df_agg[m], errors='coerce')
df_agg = (df_agg.groupby(['pipeline_label', 'xai_model', 'instance_id'], as_index=False)[METRICS]
                .mean())

print("Paarweise Wilcoxon signed-rank (zweiseitig) + Cliff's δ — gen-gemittelt, gepaart "
      'über (instance_id, xai_model).\n')
wilcoxon_all = []
for m in METRICS:
    wdf = wilcoxon_pairwise(df_agg, PIP_ORDER, m)
    wdf.insert(0, 'metric', m)
    wilcoxon_all.append(wdf)
    print(f'── {m} ──')
    display(wdf[['pipeline_a', 'pipeline_b', 'n_pairs', 'mean_a', 'mean_b',
                 'delta_mean', 'p_value', 'cliffs_d', 'magnitude']])
    print()
wilcoxon_all_df = pd.concat(wilcoxon_all, ignore_index=True)
wilcoxon_all_df.to_csv(FAITH_DIR / 'faithfulness_scale_wilcoxon.csv', index=False)
print(f'Gespeichert: {FAITH_DIR / "faithfulness_scale_wilcoxon.csv"}')

## 6 · Plot — RA/SA/VA mit CI (n≈200)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
titles = {'RA': 'RA — Rang-Genauigkeit', 'SA': 'SA — Vorzeichen-Genauigkeit',
          'VA': 'VA — Wert-Genauigkeit'}
x = np.arange(len(PIP_ORDER))
for ax, m in zip(axes, METRICS):
    means, los, his = [], [], []
    for pipeline in PIP_ORDER:
        vals = pd.to_numeric(faith_df[faith_df.pipeline_label == pipeline][m],
                             errors='coerce').dropna().values
        lo, hi, obs = bootstrap_ci(vals)
        means.append(obs); los.append(obs - lo); his.append(hi - obs)
    colors = [PIPELINE_COLORS[p] for p in PIP_ORDER]
    ax.bar(x, means, yerr=[los, his], color=colors, capsize=4, width=0.6)
    ax.set_xticks(x); ax.set_xticklabels(PIP_ORDER, fontsize=9, rotation=10)
    ax.set_ylim(0, 1.1); ax.set_title(titles[m], fontsize=10)
    ax.set_ylabel('Genauigkeit (0–1)')
    ax.axhline(1.0, color='gray', ls='--', alpha=0.35, lw=1)
plt.suptitle(f'Faithfulness (Ichmoukhamedov Gl. 1) — n≈{len(INSTANCE_IDS_SCALE)} × {N_GEN} Gen. '
             '(95 %-Bootstrap-CI)', y=1.03, fontsize=12)
plt.tight_layout()
out = FAITH_DIR / 'faithfulness_scale.png'
plt.savefig(out, dpi=130, bbox_inches='tight'); plt.show()
print(f'Gespeichert: {out}')